# Colab setup (run first)

On **Google Colab**, run the two cells below first (mount Drive + install deps), then select a **GPU runtime**. Edit `DRIVE_PROJECT_PATH` to match your upload location. On a local machine these cells are skipped automatically.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_PROJECT_PATH = "/content/drive/MyDrive/PRAgenticAI"  # edit to your upload location

    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")

    PROJECT_ROOT = Path(DRIVE_PROJECT_PATH).resolve()
    assert PROJECT_ROOT.exists(), (
        f"Project folder not found at {PROJECT_ROOT}. Upload the project to Drive "
        "and set DRIVE_PROJECT_PATH to its location."
    )
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "requirements.txt").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab only). Colab already ships CUDA-enabled PyTorch.
if IN_COLAB:
    %pip install -q \
        "transformers>=4.46.0" \
        "datasets>=2.18.0" \
        "peft>=0.13.0" \
        "accelerate>=0.34.0" \
        sentencepiece \
        "evaluate>=0.4.0" \
        "sacrebleu>=2.4.0" \
        "bert-score>=0.3.13" \
        "codebleu>=0.7.0" \
        "nltk>=3.8.0" \
        tree-sitter tree-sitter-python tree-sitter-java \
        pyyaml
    print("Dependencies installed.")
else:
    print("Not on Colab; skipping pip install.")

# Baseline Evaluation — Qwen2.5-Coder-0.5B

Evaluate the **pre–fine-tune** base model on AVATAR-TC Java→Python pairs.

Metrics: BLEU, BERTScore, CodeBLEU, CodeBERTScore (all via `evaluation/metrics.py`).

In [ ]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT  # type: ignore[used-before-def]  # noqa: F821
except NameError:
    _here = Path.cwd()
    PROJECT_ROOT = (_here if (_here / "requirements.txt").exists() else _here.parent).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.run_baseline_qwen import run_baseline_eval, print_report

## Config

Start with a small `MAX_SAMPLES` smoke test, then set to `None` for the full split.

In [ ]:
SPLIT = "valid"        # "valid" (443) or "test" (1745)
MAX_SAMPLES = 10       # None for full split
DEVICE = "auto"        # auto -> CUDA on Colab, MPS on Mac, CPU otherwise

In [ ]:
report = run_baseline_eval(
    split=SPLIT,
    max_samples=MAX_SAMPLES,
    device=DEVICE,
)
print_report(report)

## Inspect results

In [ ]:
import pandas as pd

metric_keys = [
    "bleu", "bertscore_f1", "codebleu", "code_bertscore_f1",
    "codebleu_ngram", "codebleu_syntax", "codebleu_dataflow",
]
pd.Series({k: report[k] for k in metric_keys if k in report}).round(4)

In [ ]:
# Compare one prediction vs reference
sample = report["samples"][0]
print("Java input:", sample["input_preview"])
print("\nPrediction:", sample["prediction_preview"])
print("\nReference:", sample["reference_preview"])